In [1]:
from pynq.pl_server.global_state import clear_global_state
clear_global_state()
from pynq import Overlay

overlay = Overlay("/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/VivadoHW.bit")

In [ ]:
from pynq import MMIO, allocate
import cv2
import numpy as np

vdma_info = overlay.ip_dict['axi_vdma_0']
vdma = MMIO(vdma_info['phys_addr'], vdma_info['addr_range'])

cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

width = 640
height = 480
stride = width * 3
hsize = width * 3

input_buffer = allocate(shape=(height, width, 3), dtype=np.uint8)

# 1. Soft reset & clear errors
vdma.write(0x00, 0x00000004)
vdma.write(0x04, 0xFFFFFFFF)
vdma.write(0x00, 0x00000000)

# 2. Configure buffer address, stride, and horizontal size
vdma.write(0x5C, input_buffer.physical_address)
vdma.write(0x58, stride)
vdma.write(0x54, hsize)

# 3. Start VDMA in Park Mode: Run=1, Circular=0 -> 0x00010001
vdma.write(0x00, 0x00010001)

# 4. Trigger transfer via Vertical Size
vdma.write(0x50, height)

print("Streaming active in Park Mode. Check your HDMI display!")

frame_count = 0
try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        np.copyto(input_buffer, frame)
        input_buffer.flush()
        
        frame_count += 1
        if frame_count % 30 == 0:
            status = vdma.read(0x04)
            ctrl = vdma.read(0x00)
            print(f"Frames: {frame_count} | Control: {hex(ctrl)} | Status: {hex(status)}", end="\r")
            
            if status & 0x1:
                print(f"\n[!] VDMA halted with status: {hex(status)}")
                break
        
except KeyboardInterrupt:
    vdma.write(0x00, 0x00000000)
    input_buffer.freebuffer()
    cap.release()
    print("\nStream stopped.")

: 

: 

: 